ChEMBL_Activity.csv
        │
        ▼
load_chembl_csv()
        │
        ▼
prepare_endpoint()
        │
        │  Keep:
        │  IC50
        │  relation "="
        │  units nM
        │  valid pChEMBL
        │  valid SMILES
        ▼
standardize_dataframe()
        │
        │  df["Smiles"].apply(standardize_smiles)
        │
        ▼
SMILES_STD
        │
        ▼
aggregate_replicates()
        │
        │  groupby standardized molecule
        │  median pChEMBL
        │  count replicates
        │  measure disagreement
        ▼
add_scaffolds()
        │
        │  df["SMILES_STD"].apply(calculate_scaffold)
        │
        ├────────────► scaffold QC
        │              top scaffolds
        │              molecule images
        ▼
add_descriptors()
        │
        │  df["SMILES_STD"].apply(calculate_descriptors)
        │
        ▼
16 RDKit descriptors
        │
        ▼
scaffold_train_test_split()
        │
        ├──── train scaffolds
        │
        └──── completely unseen test scaffolds
                 │
                 ▼
filter_descriptors()
        │
        │ missingness > 20%
        │ variance < 0.001
        │ |correlation| > 0.80
        ▼
Selected descriptors
        │
        ├───────────────┐
        ▼               ▼
run_random_search()   run_bayesian_search()
        │               │
        │ RandomizedCV  │ BayesSearchCV
        │               │
        └───────┬───────┘
                ▼
        choose_best_search()
                │
                ▼
      best scaffold-CV RMSE
                │
                ▼
         evaluate_model()
                │
                ▼
        untouched test set
                │
        ┌───────┼────────┐
        ▼       ▼        ▼
       RMSE     MAE       R²

In [1]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import loguniform

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.impute import SimpleImputer
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from skopt import BayesSearchCV
from skopt.space import Real

RANDOM_STATE = 42
UNCHARGER = rdMolStandardize.Uncharger()

DESCRIPTORS = {
    "ExactMolWt": Descriptors.ExactMolWt,
    "HeavyAtomMolWt": Descriptors.HeavyAtomMolWt,
    "LabuteASA": Descriptors.LabuteASA,
    "MaxAbsEStateIndex": Descriptors.MaxAbsEStateIndex,
    "MaxAbsPartialCharge": Descriptors.MaxAbsPartialCharge,
    "MaxEStateIndex": Descriptors.MaxEStateIndex,
    "MaxPartialCharge": Descriptors.MaxPartialCharge,
    "MinAbsEStateIndex": Descriptors.MinAbsEStateIndex,
    "MinAbsPartialCharge": Descriptors.MinAbsPartialCharge,
    "MinEStateIndex": Descriptors.MinEStateIndex,
    "MinPartialCharge": Descriptors.MinPartialCharge,
    "MolLogP": Descriptors.MolLogP,
    "MolMR": Descriptors.MolMR,
    "MolWt": Descriptors.MolWt,
    "TPSA": Descriptors.TPSA,
    "qed": Descriptors.qed,
}

In [2]:
dataset_path = "/home/pospim/Desktop/school/bip_chem/exer/ChEMBL_Activity.csv"
dataset = pd.read_csv(dataset_path, delimiter=";")
dataset.head
dataset.columns

/tmp/ipykernel_275964/1515981327.py:2: DtypeWarning: Columns (0: Assay Tissue ChEMBL ID, 1: Assay Tissue Name, 2: Assay Subcellular Fraction, 3: Assay Variant Accession, 4: Assay Variant Mutation) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv(dataset_path, delimiter=";")


Index(['Molecule ChEMBL ID', 'Molecule Name', 'Molecule Max Phase',
       'Molecular Weight', '#RO5 Violations', 'AlogP', 'Compound Key',
       'Smiles', 'Standard Type', 'Standard Relation', 'Standard Value',
       'Standard Units', 'pChEMBL Value', 'Data Validity Comment', 'Comment',
       'Uo Units', 'Ligand Efficiency BEI', 'Ligand Efficiency LE',
       'Ligand Efficiency LLE', 'Ligand Efficiency SEI', 'Potential Duplicate',
       'Assay ChEMBL ID', 'Assay Description', 'Assay Type', 'BAO Format ID',
       'BAO Label', 'Assay Organism', 'Assay Tissue ChEMBL ID',
       'Assay Tissue Name', 'Assay Cell Type', 'Assay Subcellular Fraction',
       'Assay Parameters', 'Assay Variant Accession', 'Assay Variant Mutation',
       'Target ChEMBL ID', 'Target Name', 'Target Organism', 'Target Type',
       'Document ChEMBL ID', 'Source ID', 'Source Description',
       'Document Journal', 'Document Year', 'Cell ChEMBL ID', 'Properties',
       'Action Type', 'Standard Text Value', 'V

In [3]:
def prepare_data(df):
    data = df.copy()

    data["relation_clean"] = (
        data["Standard Relation"]
        .astype(str)
        .str.replace("'", "", regex=False)
        .str.strip()
    )
    data["pChEMBL"] = pd.to_numeric(
        data["pChEMBL Value"],
        errors="coerce"
    )

    mask = (
        (data["Standard Type"] == "IC50")
        & (data["relation_clean"] == "=")
        & (data["Standard Units"] == "nM")
        & (data["pChEMBL"].notna())
        & (data["Smiles"].str.strip() != "")
    )

    return data.loc[mask].copy()

clean_data = prepare_data(dataset)
clean_data.head

<bound method NDFrame.head of       Molecule ChEMBL ID  Molecule Name  Molecule Max Phase  Molecular Weight  \
2          CHEMBL1276308   MIFEPRISTONE                 4.0            429.60   
3              CHEMBL131   PREDNISOLONE                 4.0            360.45   
4          CHEMBL4202668            NaN                 NaN            506.55   
5          CHEMBL4211898            NaN                 NaN            528.99   
8          CHEMBL4217184            NaN                 NaN            485.26   
...                  ...            ...                 ...               ...   
17886      CHEMBL1201396    FLUTICASONE                 3.0            444.52   
17924       CHEMBL384467  DEXAMETHASONE                 4.0            392.47   
17925      CHEMBL5812828            NaN                 NaN            608.64   
17926      CHEMBL6031431            NaN                 NaN            684.77   
17927      CHEMBL5758568            NaN                 NaN            754.90  

In [4]:
def standardize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)

        if not mol:
            return pd.Series({
                "SMILES_STD": np.nan,
                "standardization_changed": False,
                "standardization_error": "Could not parse SMILES",
            })
        mol = rdMolStandardize.Cleanup(mol)

        mol = rdMolStandardize.FragmentParent(mol)

        # Keep this cell self-contained when it is rerun out of order.
        uncharger = globals().get("UNCHARGER")
        if uncharger is None:
            uncharger = rdMolStandardize.Uncharger()
        mol = uncharger.uncharge(mol)

        Chem.SanitizeMol(mol)

        standardized = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )
        return pd.Series({
            "SMILES_STD": standardized,
            "standardization_changed":
                standardized != smiles,
            "standardization_error": "",
        })

    except Exception as exc:

        return pd.Series({
            "SMILES_STD": np.nan,
            "standardization_changed": False,
            "standardization_error": str(exc),
        })

standardization = (
    clean_data["Smiles"]
    .apply(standardize_smiles)
)

[20:12:42] Initializing MetalDisconnector
[20:12:42] Running MetalDisconnector
[20:12:42] Initializing Normalizer
[20:12:42] Running Normalizer
[20:12:42] Initializing MetalDisconnector
[20:12:42] Running MetalDisconnector
[20:12:42] Initializing Normalizer
[20:12:42] Running Normalizer
[20:12:42] Running LargestFragmentChooser
[20:12:42] Running Uncharger
[20:12:42] Initializing MetalDisconnector
[20:12:42] Running MetalDisconnector
[20:12:42] Initializing Normalizer
[20:12:42] Running Normalizer
[20:12:42] Initializing MetalDisconnector
[20:12:42] Running MetalDisconnector
[20:12:42] Initializing Normalizer
[20:12:42] Running Normalizer
[20:12:42] Running LargestFragmentChooser
[20:12:42] Running Uncharger
[20:12:42] Initializing MetalDisconnector
[20:12:42] Running MetalDisconnector
[20:12:42] Initializing Normalizer
[20:12:42] Running Normalizer
[20:12:42] Initializing MetalDisconnector
[20:12:42] Running MetalDisconnector
[20:12:42] Initializing Normalizer
[20:12:42] Running Norma

In [5]:
standardized = pd.concat(
    [
        clean_data.reset_index(drop=True),
        standardization.reset_index(drop=True),
    ],
    axis=1,
)
standardized.head()

,Molecule ChEMBL ID,Molecule Name,Molecule Max Phase,Molecular Weight,#RO5 Violations,AlogP,Compound Key,Smiles,Standard Type,Standard Relation,...,Cell ChEMBL ID,Properties,Action Type,Standard Text Value,Value,relation_clean,pChEMBL,SMILES_STD,standardization_changed,standardization_error
0,CHEMBL1276308,MIFEPRISTONE,4.0,429.60,1.0,5.41,5,CC#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=CC(=O)CCC4=C3[...,IC50,'=',...,NaN,NaN,NaN,NaN,8.7,=,8.06,CC#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=CC(=O)CCC4=C3[...,False,
1,CHEMBL131,PREDNISOLONE,4.0,360.45,0.0,1.56,Prednisolone,C[C@]12C=CC(=O)C=C1CC[C@@H]1[C@@H]2[C@@H](O)C[...,IC50,'=',...,NaN,NaN,NaN,NaN,7.0,=,8.15,C[C@]12C=CC(=O)C=C1CC[C@@H]1[C@@H]2[C@@H](O)C[...,False,
2,CHEMBL4202668,NaN,NaN,506.55,1.0,4.88,24,C[C@H](NC(=O)C(C)(F)F)[C@H](Oc1ccc2c(cnn2-c2cc...,IC50,'=',...,NaN,NaN,NaN,NaN,4.4,=,8.36,C[C@H](NC(=O)C(C)(F)F)[C@H](Oc1ccc2c(cnn2-c2cc...,False,
3,CHEMBL4211898,NaN,NaN,528.99,2.0,5.29,27,CC(C)[C@H](NC(=O)C(C)(F)F)[C@H](Oc1ccc2c(cnn2-...,IC50,'=',...,NaN,NaN,NaN,NaN,2.6,=,8.59,CC(C)[C@H](NC(=O)C(C)(F)F)[C@H](Oc1ccc2c(cnn2-...,False,
4,CHEMBL4217184,NaN,NaN,485.26,0.0,3.15,38,N#Cc1ccc(O[C@H](c2cnc(C3CC3)nc2)[C@H](CO)NC(=O...,IC50,'=',...,NaN,NaN,NaN,NaN,42.0,=,7.38,N#Cc1ccc(O[C@H](c2cnc(C3CC3)nc2)[C@H](CO)NC(=O...,False,


In [6]:
def aggregate_replicates(df):

    curated = (
        df.groupby(
            "SMILES_STD",
            as_index=False,
        )
        .agg(
            pChEMBL=("pChEMBL", "median"),

            pChEMBL_mean=(
                "pChEMBL",
                "mean",
            ),

            pChEMBL_min=(
                "pChEMBL",
                "min",
            ),

            pChEMBL_max=(
                "pChEMBL",
                "max",
            ),

            n_measurements=(
                "pChEMBL",
                "size",
            ),
        )
    )

    curated["pChEMBL_range"] = (
        curated["pChEMBL_max"]
        - curated["pChEMBL_min"]
    )

    return curated

curated = aggregate_replicates(standardized)

In [7]:
def calculate_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)

    scaffold = (
        MurckoScaffold.MurckoScaffoldSmiles(
            mol=mol,
            includeChirality=False
        )
    )
    if scaffold:
        return scaffold

    return "<ACYCLIC>"

curated["scaffold"] = (
    curated["SMILES_STD"]
    .apply(calculate_scaffold)
)

In [8]:
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    values = {}
    for name, function in DESCRIPTORS.items():
        try:
            values[name] = float(function(mol))
        except Exception:
            values[name] = np.nan

    return values

# Keep the expected feature schema even when the input is empty.
descriptor_columns = list(DESCRIPTORS.keys())
descriptor_df = pd.DataFrame(
    curated["SMILES_STD"].apply(calculate_descriptors).tolist(),
    index=curated.index,
    columns=descriptor_columns,
)

In [9]:
TEST_SIZE = 0.25

def scaffold_train_test_split(
        df, descriptor_columns, tst_size=TEST_SIZE
):
    if df.empty:
        raise ValueError(
            "No molecules are available for scaffold splitting. "
            "Rerun the standardization and aggregation cells first."
        )

    X = df[descriptor_columns].copy()
    y = df["pChEMBL"].to_numpy(dtype=float)
    groups = df["scaffold"].to_numpy()
    if pd.isna(groups).any():
        raise ValueError("Scaffold labels contain missing values.")
    if len(np.unique(groups)) < 2:
        raise ValueError("Scaffold splitting requires at least two distinct scaffolds.")

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=tst_size,
        random_state=RANDOM_STATE
    )

    train_idx, test_idx = next(
        splitter.split(
            X,y,groups=groups
        )
    )
    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y[train_idx]
    y_test = y[test_idx]

    groups_train = groups[train_idx]
    groups_test = groups[test_idx]

    assert set(groups_train).isdisjoint(set(groups_test))

    print("\n=== OUTER SCAFFOLD SPLIT ===")
    print(f"Train molecules:  {len(train_idx):,}")
    print(f"Test molecules:   {len(test_idx):,}")
    print(f"Train scaffolds:  {len(set(groups_train)):,}")
    print(f"Test scaffolds:   {len(set(groups_test)):,}")

    return (
        X_train,
        X_test,
        y_train,
        y_test,
        groups_train,
        groups_test,
        train_idx,
        test_idx,
    )

In [10]:
model_table = pd.concat(
    [
        curated.reset_index(drop=True),
        descriptor_df.reset_index(drop=True)
    ],
    axis=1
)
(
    X_train_raw,
    X_test_raw,
    y_train,
    y_test,
    groups_train,
    groups_test,
    train_idx,
    test_idx,
) = scaffold_train_test_split(
    model_table,
    list(DESCRIPTORS.keys()),
)


=== OUTER SCAFFOLD SPLIT ===
Train molecules:  1,616
Test molecules:   513
Train scaffolds:  518
Test scaffolds:   173


## Train and select the best scaffold-aware model

The reusable training and model-selection implementation is in `qsar_model_pipeline.py`. Both searches use grouped cross-validation, so a scaffold never appears in both sides of a CV fold.

In [ ]:
from qsar_model_pipeline import save_selected_model, train_and_select_best_model

training_result = train_and_select_best_model(
    X_train_raw,
    y_train,
    groups_train,
    random_state=RANDOM_STATE,
    n_iter=40,
    n_splits=5,
    n_jobs=-1,
)
display(training_result.leaderboard)

model_path = save_selected_model(
    training_result,
    Path("artifacts/best_scaffold_qsar.joblib"),
    metadata={
        "target": "pChEMBL",
        "split": "scaffold",
        "random_state": RANDOM_STATE,
    },
)
print(f"Selected {training_result.best_name}; saved fitted pipeline to {model_path}")

## Results on the held-out ChEMBL scaffold test set

This comparison is performed once on the untouched outer test set. Model selection above uses only the training set and grouped cross-validation.

In [ ]:
chembl_test_results = []
chembl_test_predictions = model_table.iloc[test_idx][
    ["SMILES_STD", "scaffold", "pChEMBL"]
].reset_index(drop=True).copy()

for model_name, search in training_result.searches.items():
    predictions = search.best_estimator_.predict(X_test_raw)
    chembl_test_results.append({
        "model": model_name,
        "selected": model_name == training_result.best_name,
        "scaffold_cv_rmse": -float(search.best_score_),
        "test_rmse": mean_squared_error(y_test, predictions) ** 0.5,
        "test_mae": mean_absolute_error(y_test, predictions),
        "test_r2": r2_score(y_test, predictions),
    })
    chembl_test_predictions[f"predicted_{model_name}"] = predictions

chembl_test_results = (
    pd.DataFrame(chembl_test_results)
    .sort_values("scaffold_cv_rmse")
    .reset_index(drop=True)
)
display(chembl_test_results)
display(chembl_test_predictions.head(20))

best_test_predictions = training_result.best_model.predict(X_test_raw)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, best_test_predictions, alpha=0.55)
limits = [min(y_test.min(), best_test_predictions.min()), max(y_test.max(), best_test_predictions.max())]
ax.plot(limits, limits, "k--", linewidth=1)
ax.set(xlabel="Observed pChEMBL", ylabel="Predicted pChEMBL", title=f"Held-out ChEMBL: {training_result.best_name}")
plt.show()

## Apply the selected model to generated molecules

Prediction remains in the notebook because loading and inspecting generated chemistry is dataset-specific. Invalid SMILES are retained with a missing prediction.

In [ ]:
def predict_generated_molecules(generated_data, smiles_column=None):
    output = generated_data.copy()
    if smiles_column is None:
        candidates = {"smiles", "canonical_smiles", "smiles_std"}
        smiles_column = next(
            (column for column in output.columns if column.strip().lower() in candidates),
            None,
        )
    if smiles_column is None or smiles_column not in output.columns:
        raise ValueError("Provide the name of the generated-data SMILES column.")

    generated_standardization = output[smiles_column].apply(standardize_smiles)
    output["SMILES_STD"] = generated_standardization["SMILES_STD"]
    output["standardization_error"] = generated_standardization["standardization_error"]
    valid = output["SMILES_STD"].notna()
    output["predicted_pChEMBL"] = np.nan

    if valid.any():
        generated_descriptors = pd.DataFrame(
            output.loc[valid, "SMILES_STD"].apply(calculate_descriptors).tolist(),
            index=output.index[valid],
        ).reindex(columns=training_result.feature_columns)
        output.loc[valid, "predicted_pChEMBL"] = training_result.best_model.predict(
            generated_descriptors
        )
    return output

GENERATED_DATA_PATH = Path("generated_data.csv")
if GENERATED_DATA_PATH.exists():
    generated_data = pd.read_csv(GENERATED_DATA_PATH)
    generated_predictions = predict_generated_molecules(generated_data)
    display(generated_predictions.sort_values("predicted_pChEMBL", ascending=False).head(25))
else:
    print(
        f"Place generated molecules in {GENERATED_DATA_PATH} or call "
        "predict_generated_molecules(your_dataframe, smiles_column='your_smiles_column')."
    )